# Phase 22 — Is "Topology Helps Localization" Real, or Also Seed Noise?

Phase 18 found topology improves pocket-center localization specifically
(+0.0289 pooled DCC), not just classification — a genuinely novel claim, since
no prior work combines trained equivariant localization with topological
features. But it was a **single run**, exactly the kind of number Phase 21
just showed can be dominated by seed variance (a 0.027 single-seed std against
a 0.044 "finding" in that case). This project's actual novel contribution
should not get an exemption from the scrutiny just applied to a less
interesting result.

**Reuses everything expensive:** Phase 15's 3 `full` (with-topology) and Phase
18's 3 `no_topo` checkpoints are loaded, not retrained. 5 more seeds are
trained per arm (8 total each), and all scoring happens on Phase 15's already
-cached benchmark features (COACH420 + HOLO4K, the full 1,149-unit
leakage-clean set) — no re-fetching, no re-featurization.

## Cell 1 — Setup, rebuild training data with true centers (Phase 14 logic)

In [ ]:
!pip install -q torch-geometric==2.6.1 biopython==1.84

In [ ]:
import pickle

import gzip, json, os, time, urllib.request, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

blob = torch.load(locate("phase10_graphs.pt"), map_location="cpu", weights_only=False)
dataset = blob["dataset"]
N_PRE = int(blob["n_pre_topo"]); TOPO_DIM = int(blob["topo_dim"]); POS_WEIGHT = float(blob["pos_weight"])

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

splits = {s: [d for d in dataset if d.split == s] for s in ("train","val","test")}
print(f"device {DEVICE} | node dim {dataset[0].x.size(1)} | " +
      " | ".join(f"{k} {len(v)}" for k,v in splits.items()))

In [ ]:
from Bio.PDB import PDBParser

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}
NON_LIGAND={"HOH","DOD","WAT","SO4","PO4","GOL","EDO","PEG","PG4","PGE","1PE","MPD","TRS",
            "ACT","ACY","FMT","CIT","MES","EPE","DMS","IMD","BME","NO3","AZI","CO3","IOD",
            "BR","CL","F","NA","K","CS","RB","LI","UNX","UNL","UNK"}
CATALYTIC_METALS={"ZN","MG","MN","FE","FE2","CU","CU1","CA","NI","CO","CD","W","MO"}
MIN_LIG_ATOMS = 6
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

need = sorted({d.pdb_id for d in dataset})
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=16) as ex: got = list(ex.map(fetch, need))
paths = {p: f for p, f in got if f}
print(f"fetched {len(paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

def ligands_and_ca(pid, path, chain_id, n_expect):
    from Bio.PDB import MMCIFParser
    try:
        parser = MMCIFParser(QUIET=True) if ".cif" in path.name else PDBParser(QUIET=True, PERMISSIVE=True)
        with gzip.open(path,"rt") as fh:
            model = next(parser.get_structure(pid, fh).get_models())
        ch = model[chain_id]
    except Exception:
        return None
    residues=[r for r in ch if r.get_resname().strip().upper() in AA3_TO_1 and "CA" in r]
    if len(residues) != n_expect: return None
    raw_ca = np.asarray([r["CA"].coord for r in residues], dtype=np.float32)

    ligs = []
    for c2 in model:
        for res in c2:
            if res.id[0] == " ": continue
            nm = res.get_resname().strip().upper()
            if nm in AA3_TO_1 or nm in NON_LIGAND: continue
            xyz = np.asarray([a.coord for a in res.get_unpacked_list() if a.element not in ("H","D")],
                             dtype=np.float32).reshape(-1,3)
            is_metal = nm in CATALYTIC_METALS
            if is_metal or len(xyz) >= MIN_LIG_ATOMS: ligs.append(xyz)
    return raw_ca, ligs

t0=time.time(); n_ok=0; n_geom_fail=0; n_no_ligand=0
for d in dataset:
    if d.pdb_id not in paths: continue
    r = ligands_and_ca(d.pdb_id, paths[d.pdb_id], d.chain_id, int(d.num_nodes))
    if r is None: continue
    raw_ca, ligs = r
    if not ligs: n_no_ligand += 1; continue

    # geometry-drift check: pairwise CA distances must match d.pos (residue ORDER, not just count)
    k = min(50, len(raw_ca))
    a = np.linalg.norm(raw_ca[:k,None]-raw_ca[None,:k], axis=-1)
    b = np.linalg.norm(d.pos.numpy()[:k,None]-d.pos.numpy()[None,:k], axis=-1)
    if float(np.abs(a-b).max()) > 1e-2:
        n_geom_fail += 1; continue

    offset = raw_ca.mean(axis=0)                       # same centering d.pos already used
    y_np = d.y.numpy().astype(bool)
    contact_centroid = raw_ca[y_np].mean(axis=0) if y_np.any() else raw_ca.mean(axis=0)
    # "primary" ligand = the one whose center of mass is nearest the labelled contact residues
    coms = [lig.mean(axis=0) for lig in ligs]
    best = int(np.argmin([np.linalg.norm(c - contact_centroid) for c in coms]))
    d.true_center = torch.from_numpy((coms[best] - offset).astype(np.float32))
    n_ok += 1

print(f"\ntrue centers recovered: {n_ok:,}/{len(dataset):,} "
      f"({(time.time()-t0)/60:.1f} min) | geometry mismatches: {n_geom_fail} | no ligand: {n_no_ligand}")

dataset = [d for d in dataset if hasattr(d, "true_center")]
print(f"training chains with recovered true centers: {len(dataset):,}")

rng = np.random.RandomState(42)
perm = rng.permutation(len(dataset))
n_val = max(1, int(0.08 * len(perm)))
splits = {"val": [dataset[i] for i in perm[:n_val]], "train": [dataset[i] for i in perm[n_val:]]}
n_tr, n_va = len(splits["train"]), len(splits["val"])
print(f"deployment training: {n_tr} train / {n_va} val  (matches Phase 15/18's 2,056-chain setup)")

## Cell 2 — Model (identical to Phase 14/15/18)

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 3 — Training machinery (identical to Phase 14)

In [ ]:
def run_epoch(model, loader, opt=None, lam=0.0):
    tr = opt is not None; model.train(tr)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    huber = nn.SmoothL1Loss(beta=4.0)                    # ~matches the 4A DCC threshold
    ps, ys, cd = [], [], []
    for b in loader:
        b = b.to(DEVICE)
        with torch.set_grad_enabled(tr):
            logits, pred_c = model(b)
            loss = crit(logits, b.y)
            if lam > 0 and pred_c is not None:
                true_c = b.true_center.view(-1, 3)
                loss = loss + lam * huber(pred_c, true_c)
                cd.append((pred_c - true_c).norm(dim=1).detach().cpu().numpy())
        if tr:
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        ps.append(torch.sigmoid(logits).detach().cpu().numpy()); ys.append(b.y.cpu().numpy())
    p, y = np.concatenate(ps), np.concatenate(ys)
    out = {"pr": average_precision_score(y, p), "roc": roc_auc_score(y, p)}
    if cd:
        d_ = np.concatenate(cd)
        out["dcc"] = float((d_ < 4.0).mean()); out["mean_center_err"] = float(d_.mean())
    return out


def fit(use_loc_head, lam, seed, epochs=40, patience=8, batch=16):
    torch.manual_seed(seed); np.random.seed(seed)
    model = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=use_loc_head).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tl = DataLoader(splits["train"], batch_size=batch, shuffle=True)
    vl = DataLoader(splits["val"], batch_size=batch)
    best, state, bad = -1, None, 0
    for ep in range(epochs):
        run_epoch(model, tl, opt, lam)
        v = run_epoch(model, vl, lam=lam)
        sch.step()
        if v["pr"] > best:
            best, state, bad = v["pr"], {k:t.clone() for k,t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); model.eval()
    return model, best

## Cell 4 — Load existing checkpoints, train 5 more seeds per arm (8 total)

Topology is zeroed for the `no_topo` arm during training, exactly as Phase 18
did it -- same architecture, same budget, only the input differs.

In [ ]:
import time
WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)

def load_existing(prefix, seeds):
    out = {}
    for seed in seeds:
        cands = []
        for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
            if r.exists(): cands.extend(r.rglob(f"{prefix}{seed}.pt"))
        if not cands: continue
        state = torch.load(max(cands, key=lambda p: p.stat().st_size), map_location=DEVICE)
        m = LocalizingEGNN(N_PRE, TOPO_DIM).to(DEVICE); m.load_state_dict(state); m.eval()
        out[seed] = m
    return out

full_models = load_existing("loc_deploy_seed", (0,1,2))
notopo_models = load_existing("notopo_deploy_seed", (0,1,2))
print(f"loaded {len(full_models)} 'full' + {len(notopo_models)} 'no_topo' existing checkpoints")

ORIG_X = [d.x.clone() for d in dataset]
def zero_topology():
    for d, x0 in zip(dataset, ORIG_X):
        t = x0[:, N_PRE:].clone(); t.zero_()
        d.x = torch.cat([x0[:, :N_PRE], t], dim=1)
def restore_topology():
    for d, x0 in zip(dataset, ORIG_X):
        d.x = x0.clone()

BEST_LAMBDA = 0.1
t0 = time.time()

for seed in (3,4,5,6,7):
    restore_topology()
    m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
    v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
    full_models[seed] = m
    torch.save(m.state_dict(), WORK/f"cache/loc_deploy_seed{seed}.pt")
    print(f"  [full] seed {seed}: val PR {val_pr:.4f}  val DCC {v.get('dcc',float('nan')):.4f}  "
          f"[{(time.time()-t0)/60:.1f}m]")

for seed in (3,4,5,6,7):
    zero_topology()
    m, val_pr = fit(use_loc_head=True, lam=BEST_LAMBDA, seed=seed)
    v = run_epoch(m, DataLoader(splits["val"], batch_size=16), lam=BEST_LAMBDA)
    notopo_models[seed] = m
    torch.save(m.state_dict(), WORK/f"cache/notopo_deploy_seed{seed}.pt")
    print(f"  [no_topo] seed {seed}: val PR {val_pr:.4f}  val DCC {v.get('dcc',float('nan')):.4f}  "
          f"[{(time.time()-t0)/60:.1f}m]")

restore_topology()
print(f"\n{len(full_models)} 'full' + {len(notopo_models)} 'no_topo' seeds available (0-7 each)")

## Cell 5 — Load Phase 15's cached benchmark features (COACH420 + HOLO4K, no re-fetch)

In [ ]:
def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

with open(locate("phase15_raw_features.pkl"), "rb") as fh:
    raw_cache = pickle.load(fh)
print(f"loaded cached features for {len(raw_cache):,} chains (COACH420 + HOLO4K, no re-featurization)")

blob6b_bench = torch.load(locate("phase6b_graphs.pt"), map_location="cpu", weights_only=False)
mu_b, sd_b = blob6b_bench["topo_mean"], blob6b_bench["topo_std"]

## Cell 6 — Per-seed and per-ensemble-size DCC, both arms

In [ ]:
from collections import defaultdict

@torch.no_grad()
def predict_all_seeds(entry, model_dict, zero_topo=False):
    x = torch.from_numpy(entry["x"]).clone()
    if zero_topo:
        x[:, N_PRE:] = 0.0
    else:
        x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu_b)) / torch.from_numpy(sd_b)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=torch.from_numpy(entry["pos"]).to(DEVICE)
    d.edge_index=torch.from_numpy(entry["edge_index"]).to(DEVICE)
    d.edge_attr=torch.from_numpy(entry["edge_attr"]).to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    offset = entry["ca"].mean(axis=0)
    out = {}
    for seed, m in model_dict.items():
        lg, pc = m(d)
        out[seed] = {"prob_sum": float(torch.sigmoid(lg).sum().cpu()),
                     "center": pc.cpu().numpy()[0] + offset}
    return out


def dcc(pred_center, ligs):
    if pred_center is None or not ligs: return None
    best = np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        d_ = float(np.linalg.norm(pred_center - lig_atoms.mean(axis=0)))
        best = min(best, d_)
    return best if np.isfinite(best) else None


t0 = time.time()
by_structure = defaultdict(list)
for key, entry in raw_cache.items():
    by_structure[key[:2]].append((key[2], entry))

full_out, notopo_out, ligs_of = {}, {}, {}
for (src_, pid), chain_list in by_structure.items():
    ligs_of[(src_,pid)] = chain_list[0][1]["ligs"]
    full_out[(src_,pid)] = [(cid, predict_all_seeds(e, full_models, zero_topo=False)) for cid, e in chain_list]
    notopo_out[(src_,pid)] = [(cid, predict_all_seeds(e, notopo_models, zero_topo=True)) for cid, e in chain_list]
print(f"inference (8 seeds x 2 arms x {len(by_structure)} structures) in {(time.time()-t0)/60:.1f} min")


def dcc_rate_for_seedset(chain_output_dict, seeds):
    hits, n = 0, 0
    for key, chain_list in chain_output_dict.items():
        best_conf, best_center = -np.inf, None
        for cid, per_seed in chain_list:
            centers = [per_seed[s]["center"] for s in seeds if s in per_seed]
            confs = [per_seed[s]["prob_sum"] for s in seeds if s in per_seed]
            if not centers: continue
            c = np.mean(centers, axis=0); conf = np.mean(confs)
            if conf > best_conf: best_conf, best_center = conf, c
        d_ = dcc(best_center, ligs_of[key])
        if d_ is None: continue
        n += 1; hits += int(d_ < 4.0)
    return hits / max(n, 1), n


print(f"\n{'per-seed DCC':<20}{'full (topo)':>14}{'no_topo':>14}{'delta':>10}")
print("-"*58)
full_seed_dccs, notopo_seed_dccs = [], []
for seed in range(8):
    fd, _ = dcc_rate_for_seedset(full_out, (seed,))
    nd, _ = dcc_rate_for_seedset(notopo_out, (seed,))
    full_seed_dccs.append(fd); notopo_seed_dccs.append(nd)
    print(f"seed {seed:<15}{fd:>14.4f}{nd:>14.4f}{fd-nd:>10.4f}")

full_seed_dccs = np.array(full_seed_dccs); notopo_seed_dccs = np.array(notopo_seed_dccs)
print(f"\nfull:    mean {full_seed_dccs.mean():.4f}  std {full_seed_dccs.std():.4f}")
print(f"no_topo: mean {notopo_seed_dccs.mean():.4f}  std {notopo_seed_dccs.std():.4f}")
print(f"per-seed paired delta: mean {(full_seed_dccs-notopo_seed_dccs).mean():+.4f}  "
      f"std {(full_seed_dccs-notopo_seed_dccs).std():.4f}  "
      f"wins {(full_seed_dccs>notopo_seed_dccs).sum()}/8")

## Cell 7 — Ensemble-size sweep for both arms

In [ ]:
print(f"{'size':<8}{'full mean':>12}{'no_topo mean':>14}{'delta':>10}")
print("-"*44)
sweep = {}
for size in (1, 3, 5, 8):
    seeds = tuple(range(size)) if size < 8 else tuple(range(8))
    fd, fn = dcc_rate_for_seedset(full_out, seeds)
    nd, nn = dcc_rate_for_seedset(notopo_out, seeds)
    sweep[size] = {"full": fd, "no_topo": nd, "delta": fd - nd, "n_full": fn, "n_notopo": nn}
    print(f"{size:<8}{fd:>12.4f}{nd:>14.4f}{fd-nd:>10.4f}")

full8, n8f = dcc_rate_for_seedset(full_out, tuple(range(8)))
notopo8, n8n = dcc_rate_for_seedset(notopo_out, tuple(range(8)))
delta8 = full8 - notopo8
noise = full_seed_dccs.std()

print(f"\n8-seed delta: {delta8:+.4f}  vs single-seed noise ({noise:.4f}): {abs(delta8)/max(noise,1e-9):.2f}x")
print(f"Phase 18's single-run (3 seed) result was: +0.0289 pooled DCC")

verdict = ("CONFIRMED: topology's localization benefit survives proper seed scrutiny"
          if delta8 > 0.5*noise else
          "NOT CONFIRMED at this n: the effect is within plausible seed noise, "
          "same caveat as the P2Rank gap before Phase 21")
print(f"\nVERDICT: {verdict}")

json.dump({"per_seed": {"full": full_seed_dccs.tolist(), "no_topo": notopo_seed_dccs.tolist()},
           "sweep": sweep, "delta_8seed": delta8, "noise_scale": float(noise), "verdict": verdict},
          open(WORK/"phase22_summary.json", "w"), indent=2)
print("\nsaved -> phase22_summary.json")

---

## Reading this

Same standard as Phase 21: the 8-seed delta judged against the single-seed
noise floor, not the raw number alone. If the effect survives at several
times the noise scale, "topology helps localization" is a properly validated,
citable finding — the actual unique contribution of this project, not just a
plausible-sounding one. If it doesn't survive, that is equally worth knowing
before it goes into a writeup as a headline claim.